In [1]:
%%python --version
pass

Python 3.13.7


In [3]:
import gc
import math
import random
import resource
import time
from multiprocessing import Pool

import networkx as nx
import pandas as pd
import numpy as np

from Influence_Modelling import Metrics, Sampling, XsvNetwork


def concatenate_individual_runs(rows: pd.DataFrame) -> pd.DataFrame:
    combined = pd.DataFrame.from_dict(
        data={
            "dataset": [rows["dataset"].to_string().split(" ")[-1]],
            "method": [rows["method"].to_string().split(" ")[-1]],
            "nodes": [rows["nodes"].to_numpy().mean()],
            "nodes_std": [rows["nodes"].to_numpy().std()],
            "edges": [rows["edges"].to_numpy().mean()],
            "edges_std": [rows["edges"].to_numpy().std()],
            "mean_degree": [rows["mean_degree"].to_numpy().mean()],
            "mean_degree_std": [rows["mean_degree"].to_numpy().std()],
            "cc": [rows["cc"].to_numpy().mean()],
            "cc_std": [rows["cc"].to_numpy().std()],
            "centralization": [rows["centralization"].to_numpy().mean()],
            "centralization_std": [rows["centralization"].to_numpy().std()],
            "diameter": [rows["diameter"].to_numpy().std],
            "diameter_std": [rows["diameter"].to_numpy().std],
            "time_taken_ns": [rows["time_taken"].to_numpy().mean()],
            "time_taken_std_ns": [rows["time_taken"].to_numpy().std()],
            "maximum_memory_kb": [rows["maximum_memory"].to_numpy().mean()],
            "maximum_memory_std_kb": [rows["maximum_memory"].to_numpy().std()],
        },
    )
    return combined

def single_row(
    dataset_name, dataset_network, method_name, method_function, nodes
) -> pd.DataFrame:
    gc.disable()
    memory_start: resource.struct_rusage = resource.getrusage(resource.RUSAGE_SELF)
    start_time = time.time_ns()
    sample: nx.DiGraph = nx.empty_graph()
    if method_function is None:
        sample = dataset_network
    else:
        sample = method_function(dataset_network, nodes)
    result = pd.DataFrame(
        data={
            "nodes": [Metrics.s_1_number_nodes(sample)],
            "edges": [Metrics.s_2_number_edges(sample)],
            "mean_degree": [Metrics.s_3_mean_degree(sample)],
            "cc": [Metrics.s_4_clustering_coefficient(sample)],
            "centralization": [Metrics.s_6_centralization(sample)],
            "diameter": [Metrics.s_5_diameter(sample)],
        }
    )
    memory_end: resource.struct_rusage = resource.getrusage(resource.RUSAGE_SELF)
    gc.enable()
    end_time = time.time_ns()
    result["time_taken"] = end_time - start_time
    result["maximum_memory"] = memory_end.ru_maxrss - memory_start.ru_maxrss
    result["dataset"] = dataset_name
    result["method"] = method_name
    return result

In [4]:
    pd.options.display.max_columns = None
    pd.options.display.max_rows = None
    pd.options.display.expand_frame_repr = False

    # Set all parameters here
    trials: int = 2
    nodes: int = 100
    threads: int = 12

    output_file_path: str = f"sampling_results{math.floor(random.random() * 1000)}.txt"

    enron_data = XsvNetwork.XsvNetwork("", "", "enron")
    twitter_data = XsvNetwork.XsvNetwork("", "", "twitter")
    p2p_data = XsvNetwork.XsvNetwork("", "", "p2p")
    datasets = [
        ("Enron", enron_data.get_nx()),
        # ("Twitter", twitter_data.get_nx()),
        ("P2P", p2p_data.get_nx()),
    ]

    methods = [
        #("None", None),
        # ("BFS", Sampling.bfs_sample),
        # ("SNS", Sampling.snowball_sample),
        # ("RW", Sampling.random_walk_sample),
        ("MHRW", Sampling.mhrw_sample),
        ("MHDA", Sampling.mhda_sample),
        # ("Albatross", Sampling.albatross_sample),
    ]

    runs = []
    p = Pool(threads)

    for dataset_name, dataset_network in datasets:
        for method_name, method_function in methods:
            for _ in range(trials):
                runs.append(
                    (dataset_name, dataset_network, method_name, method_function, nodes)
                )

    rows = []
    all_rows = p.starmap(single_row, runs)

    # split the rows based on dataset into dataframes
    df = pd.concat(all_rows)
    compressed = pd.DataFrame()
    for method_name, _ in methods:
        for dataset_name, _ in datasets:
            combo = df[(df['method']==method_name) & (df['dataset'] == dataset_name)]
        


KeyError: None

In [4]:
df

,nodes,edges,mean_degree,cc,centralization,time_taken,maximum_memory,dataset,method
0,100,199,3.98,0.522903,0.015074,14269665,2560,Enron,MHRW
0,100,205,4.10,0.508228,0.017122,15033156,2560,Enron,MHRW
0,100,200,4.00,0.655805,0.024596,21092158,2560,Enron,MHDA
0,100,216,4.32,0.521864,0.021178,16481334,2944,Enron,MHDA


In [12]:
new = pd.DataFrame()
for method_name, _ in methods:
    for dataset_name, _ in datasets:
        new = pd.concat([concatenate_individual_runs(df[(df['method']==method_name) & (df['dataset'] == dataset_name)]), new])

In [13]:
new

,dataset,method,nodes,nodes_std,edges,edges_std,mean_degree,mean_degree_std,cc,cc_std,centralization,centralization_std,time_taken_ns,time_taken_std_ns,maximum_memory_kb,maximum_memory_std_kb
0,P2P,MHDA,100.0,0.0,105.5,0.5,2.11,0.01,0.020000,0.020000,0.016602,0.000281,6599938.0,947503.0,2624.0,64.0
0,Enron,MHDA,100.0,0.0,198.5,38.5,3.97,0.77,0.443421,0.014507,0.035532,0.017855,17537862.5,2132101.5,2624.0,64.0
0,P2P,MHRW,100.0,0.0,99.5,0.5,1.99,0.01,0.008333,0.008333,0.015051,0.004747,8568197.5,841488.5,2624.0,64.0
0,Enron,MHRW,100.0,0.0,206.0,19.0,4.12,0.38,0.477363,0.016575,0.023351,0.002305,24505816.5,21005.5,2688.0,0.0


In [15]:
tables = []
for dataset_name, _ in datasets:
    tables.append(new[new["dataset"] == dataset_name])
tables[0]

,dataset,method,nodes,nodes_std,edges,edges_std,mean_degree,mean_degree_std,cc,cc_std,centralization,centralization_std,time_taken_ns,time_taken_std_ns,maximum_memory_kb,maximum_memory_std_kb
0,Enron,MHDA,100.0,0.0,198.5,38.5,3.97,0.77,0.443421,0.014507,0.035532,0.017855,17537862.5,2132101.5,2624.0,64.0
0,Enron,MHRW,100.0,0.0,206.0,19.0,4.12,0.38,0.477363,0.016575,0.023351,0.002305,24505816.5,21005.5,2688.0,0.0
